load numpy and create a random number generator. setting the seed makes the result reproducible

In [6]:
import numpy as np
from functools import reduce
from math import lgamma, log, exp

rng = np.random.default_rng(20251013)

write the function `create_roster`. the input `n` to the `create_roster` function is the size of the roster. each call to `rng.integers` draws a ball from the bucket and returns it, so birthdays are sampled with replacement. the upper bound (366) is excluded.

In [8]:
def create_roster(n, rng=rng):
    return rng.integers(1, 366, size=n)

In [14]:
def duplicated(roster):
    return [x in roster[:i] for i, x in enumerate(roster)]

duplicated([5, 3, 8, 5, 2])

[False, False, False, True, False]

python's built-in `any` function returns `True` if any elements of the input are `True`. The `check_roster` function is simply the composition of `duplicated` and `any`.

In [15]:
def check_roster(roster):
    return any(duplicated(roster))

check_roster([5, 3, 8, 5, 2])

True

python does not have a pipe operator like `|>` in R, but we can write a small function that plays the same role. the `pipe` function takes a starting value and a se    uence fo functions, and passes the value through each function in turn.

In [16]:
def pipe(value, *functions):
    return reduce(lambda acc, f: f(acc), functions, value)

combining `create_roster` and `check_roster` with `pipe` will create a roster and immediately check if there are any duplicated birthdays.

In [17]:
pipe(35, create_roster, check_roster)

True

we can package one complete replicate (create a roster, then check it) as its own function.

In [18]:
def one_trial(n):
    return pipe(n, create_roster, check_roster)

to repeat a trial many times, we write a `replicate` function. it is a higher-order function: one of its inputs, `f`, is itself a function, which `replicate` calls `R` times. 

In [19]:
def replicate(R, f, *args):
    return [f(*args) for _ in range(R)]

replicate(10, one_trial, 35)

[True, True, True, True, True, False, True, False, True, True]

the final step is to calculate the proportion of `True`s. we could tabulate the number of `True`s and `False`s and then divide by the number of replicates. Or, because `True`s and `False`s can be treated like 1s and 0s, we can calculate the proportion by calculating the average.

In [20]:
out = replicate(1000, one_trial, 35)
np.mean(out)

np.float64(0.806)

In [21]:
def simulate_birthday_problem(n, R=10000):
    return np.mean(replicate(R, one_trial, n))

simulate_birthday_problem(20)

np.float64(0.4081)

python's `math` module does not have a log-factorial function, but it does have `lgamma`, the log of the gamma function, and $(k!) = $`lgamma(k + 1)`. so we write a small `lfactorial` helper and build `bp` from it.

In [22]:
def lfactorial(k):
    return lgamma(k + 1)

def bp(n):
    return 1 - exp(lfactorial(365) - lfactorial(365 - n) - n * log(365))

bp(35)

0.8143832388746837

simulation error:
* 35 is the class size in this example

In [23]:
np.mean(out) - bp(35)

np.float64(-0.008383238874683618)